# DataCopilot edge cases: what happens when real data goes wrong

The first notebook showed DataCopilot on a healthy dataset. This one does the opposite: it takes a **real copy of COCO128** (128 real photos)
and breaks it on purpose, one defect at a time, then runs the real DataCopilot command and shows what comes back. Most cases are
*expected failures*: the command must refuse, warn, or fail the check, and we show that by printing its **exit code** (0 means success,
1 means it ran and said no, 2 means you used it wrongly), never by leaving an error on the page.

Each case has the same shape: a headline **What could go wrong**, the code that creates the defect, the command and its real output, and a short
**What this shows**. The last cell collects every case in one table of *expected* against *actual* result, with an automatic check that they agree.

The photos are real COCO photographs (COCO annotations are CC BY 4.0; each image keeps its original Flickr licence). They are downloaded at run time into a
cache folder outside the repository and are never redistributed. The first notebook has more on the licence.

### Set-up

As in the first notebook, we prepare a clean work folder and a helper that runs a real `datacopilot` command and prints its output and exit code.
(DataCopilot also writes technical JSON log lines to the "error" stream; the helper hides them so you see only the readable messages.)

In [1]:
import hashlib
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import tempfile
import time
import types
import urllib.request
import warnings
import zipfile
from pathlib import Path

# Where downloads are cached (outside the repository) and where this notebook works.
DATA = Path(
    os.environ.get("DATACOPILOT_NOTEBOOK_DATA")
    or Path(tempfile.gettempdir()) / "datacopilot-notebooks"
).resolve()
WORK = DATA / "work-edge-cases"
shutil.rmtree(WORK, ignore_errors=True)  # start clean, so the notebook can be re-run from the top
WORK.mkdir(parents=True)
os.chdir(WORK)  # everything below uses paths relative to this folder

# Printing helper: hide the folders of this machine behind short placeholders.
_HIDE = [
    (str(p), name)
    for p, name in [
        (WORK, "<work>"),
        (WORK.resolve(), "<work>"),
        (DATA, "<data>"),
        (DATA.resolve(), "<data>"),
        (Path(tempfile.gettempdir()), "<tmp>"),
        (Path(tempfile.gettempdir()).resolve(), "<tmp>"),
        (Path.home(), "~"),
    ]
]
_HIDE.sort(key=lambda pair: -len(pair[0]))


def tidy(text):
    for real, placeholder in _HIDE:
        text = text.replace(real, placeholder)
    return text


class Done(types.SimpleNamespace):
    """What a command returned (shows nothing when it ends a cell)."""

    def _ipython_display_(self):
        pass


def datacopilot(*args, env=None, timed=False):
    """Run one `datacopilot` command like a terminal: show the command, what it prints and its exit code."""
    start = time.perf_counter()
    done = subprocess.run(  # noqa: S603  (our own command, no shell)
        [sys.executable, "-m", "datacopilot", *map(str, args)],
        capture_output=True,
        text=True,
        timeout=3600,
        env={**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", **(env or {})},
    )
    seconds = time.perf_counter() - start
    done.messages = [
        tidy(line) for line in done.stderr.splitlines() if line.strip() and not line.startswith("{")
    ]
    print("$ datacopilot " + " ".join(shlex.quote(str(a)) for a in args))
    for line in done.stdout.splitlines():
        print(tidy(line)[:240])
    for line in done.messages:
        print("[message]", line[:400])
    print(f"[exit code {done.returncode}" + (f", {seconds:.1f} s]" if timed else "]"))
    return Done(
        returncode=done.returncode,
        stdout=done.stdout,
        stderr=done.stderr,
        messages=done.messages,
        seconds=seconds,
    )


def latest_report(name="report.json"):
    """The report the last audit wrote next to us (JSON version)."""
    return json.loads(Path(name).read_text())


os.environ["DATACOPILOT_LOG_LEVEL"] = "ERROR"  # keep the technical JSON log lines out of this page
os.environ["TQDM_DISABLE"] = os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = (
    "1"  # no progress bars on this page
)
warnings.filterwarnings("ignore")

from collections import Counter

from IPython.display import Markdown, display
from PIL import Image as PILImage

import datacopilot as dc

# The scoreboard for the final table: one row per case, filled in by the cells below.
ROWS = []


def record(case, expected, ok, actual):
    """Add one row: what we expected, what really happened, and whether they agree."""
    ROWS.append((case, expected, actual, "ok" if ok else "MISMATCH"))


def said(done, limit=110):
    """The first readable message a command printed (or what it printed on standard output)."""
    text = (
        done.messages[0]
        if done.messages
        else (done.stdout.strip().splitlines() or ["(no message)"])[0]
    )
    return text if len(text) <= limit else text[: limit - 3] + "..."


def copy_of(source, name):
    """A fresh, complete copy of a dataset folder to damage."""
    shutil.rmtree(name, ignore_errors=True)
    shutil.copytree(source, name, symlinks=True)
    return Path(name)


def edit_json(path, change):
    """Load a JSON file, let `change` modify it, write it back."""
    path = Path(path)
    document = json.loads(path.read_text())
    change(document)
    path.write_text(json.dumps(document))


print("work folder:", tidy(str(WORK)), "| helpers ready")

work folder: <work> | helpers ready


**What this shows.** The helpers `record` (adds a row to the final table), `said` (the first message a command printed), `copy_of` (a fresh full copy of a dataset to damage) and
`edit_json` are all this notebook needs. Paths printed below are relative to the work folder, with `<work>` and `<data>` standing in for real folders.

## Get the real photos and make three healthy starting points

Healthy starting points are needed so that every defect is the *only* thing wrong. We download COCO128, arrange it as a YOLO dataset (103 training and 25 validation photos, as
in the first notebook), and convert it once to COCO, CVAT and Label Studio files so that we can damage whichever format a case needs.

**Where the checksum comes from, and how to update it.** The SHA-256 pinned in the next cell (and the size, 6,983,030 bytes) was taken from a first download and cross-checked: GitHub's release
listing (`gh api repos/ultralytics/assets/releases/tags/v0.0.0`) gives the same size but no digest for this old file, so the independent check was a second download over a separate connection, which gave
the identical hash. If Ultralytics ever replaces the file, the notebook stops with a mismatch error and deletes it; after you have looked at the new file, update `COCO128_SHA256` and `COCO128_BYTES`.
The archive is also treated as untrusted: every entry is checked (no absolute paths, no `..`, no links, limited size, file count and compression ratio) before anything is written, and files go into a fresh folder.
The model needs no pin here: DataCopilot's own registry fixes DINOv2-small to one exact commit and verifies the SHA-256 of its weight file (a `.safetensors` file, which holds only numbers and cannot run code) before loading it.
This notebook never loads a pickle-based `.pt` file; such files can run code when opened, so if you ever add one, verify its checksum first and prefer `.safetensors` or a loader with `weights_only=True`.

In [2]:
# Pinned fingerprint of the one file we download (see the note above this cell for where it came from).
COCO128_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip"
COCO128_SHA256 = "61e5e3028863d8ffc3b81d6a514603954889f0edd5e4b44c4ce60b2da99aeb8e"
COCO128_BYTES = 6_983_030
MAX_DOWNLOAD_BYTES = 50_000_000  # refuse anything bigger than this, whatever the server says
MAX_FILES, MAX_UNPACKED_BYTES, MAX_RATIO = 1_000, 100_000_000, 20


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def fetch_coco128():
    """Download COCO128 (HTTPS only, bounded, atomic); return its path once the pinned checksum matches."""
    if not COCO128_URL.startswith("https://"):
        raise RuntimeError("downloads must use HTTPS")
    DATA.mkdir(parents=True, exist_ok=True)
    target = DATA / "coco128.zip"
    if target.exists() and sha256_of(target) != COCO128_SHA256:
        target.unlink()  # never reuse a cached file whose checksum is wrong
    if not target.exists():
        partial = target.with_suffix(".part")
        try:
            with (
                urllib.request.urlopen(COCO128_URL, timeout=60) as reply,
                open(partial, "wb") as out,
            ):  # noqa: S310
                if int(reply.headers.get("Content-Length") or 0) > MAX_DOWNLOAD_BYTES:
                    raise RuntimeError("the server announced a file that is too big")
                received = 0
                for block in iter(lambda: reply.read(1 << 20), b""):
                    received += len(block)
                    if received > MAX_DOWNLOAD_BYTES:
                        raise RuntimeError("download is larger than the allowed maximum")
                    out.write(block)
        except OSError as problem:
            partial.unlink(missing_ok=True)
            raise RuntimeError(
                "Could not download COCO128 (" + type(problem).__name__ + "). "
                "These notebooks need the network once to fetch about 7 MB from "
                + COCO128_URL
                + ". "
                "Connect and run this cell again, or download that file yourself and save it as "
                "'coco128.zip' in the cache folder (set DATACOPILOT_NOTEBOOK_DATA to choose where it is)."
            ) from None
        except RuntimeError:
            partial.unlink(missing_ok=True)
            raise
        actual = sha256_of(partial)
        if actual != COCO128_SHA256:
            partial.unlink()
            raise RuntimeError(
                "COCO128 checksum mismatch (got "
                + actual
                + "); the file was deleted and nothing was unpacked"
            )
        partial.replace(target)  # atomic: a half-written file never carries the final name
    return target


def unpack_safely(archive_path, destination):
    """Unpack a zip we do not trust: check every entry first, then write into a fresh folder."""
    destination = Path(destination)
    if destination.exists():
        raise RuntimeError("the destination must be a fresh folder")
    with zipfile.ZipFile(archive_path) as archive:
        members = archive.infolist()
        if len(members) > MAX_FILES or sum(m.file_size for m in members) > MAX_UNPACKED_BYTES:
            raise RuntimeError("archive has too many files or unpacks to too much data")
        for member in members:
            name = member.filename
            is_link = ((member.external_attr >> 16) & 0o170000) == 0o120000
            if (
                name.startswith(("/", chr(92)))
                or ".." in Path(name).parts
                or ":" in name
                or is_link
            ):
                raise RuntimeError("unsafe entry in the archive: " + name)
            if member.compress_size and member.file_size / member.compress_size > MAX_RATIO:
                raise RuntimeError("suspiciously high compression ratio for " + name)
        for member in members:  # every entry passed; now write them one by one
            path = destination / member.filename
            if member.is_dir():
                path.mkdir(parents=True, exist_ok=True)
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source_file, open(path, "wb") as out:
                shutil.copyfileobj(source_file, out)
    return destination


zip_path = fetch_coco128()
print("file    :", zip_path.name, f"({zip_path.stat().st_size / 1e6:.1f} MB)")
print("sha256  :", sha256_of(zip_path))
print(
    "checksum matches the pinned value: True (a mismatch stops the notebook before anything is unpacked)"
)

raw = unpack_safely(zip_path, "raw")
print("images  :", len(list((raw / "coco128/images/train2017").glob("*.jpg"))))
print("labels  :", len(list((raw / "coco128/labels/train2017").glob("*.txt"))))

file    : coco128.zip (7.0 MB)
sha256  : 61e5e3028863d8ffc3b81d6a514603954889f0edd5e4b44c4ce60b2da99aeb8e
checksum matches the pinned value: True (a mismatch stops the notebook before anything is unpacked)
images  : 128
labels  : 128


**What this shows.** The same file as in the first notebook, checked by its SHA-256 fingerprint. If you are offline and have no cached copy, this cell stops with a message that says what to do.

In [3]:
# COCO128 is one folder of 128 photos with no list of class names and no validation split, so we
# add both: the 80 official COCO class names, and every fifth photo (sorted by name) becomes "val".
NAMES = [
    n.replace("_", " ")
    for n in "person bicycle car motorcycle airplane bus train truck boat traffic_light fire_hydrant stop_sign parking_meter bench bird cat dog horse sheep cow elephant bear zebra giraffe backpack umbrella handbag tie suitcase frisbee skis snowboard sports_ball kite baseball_bat baseball_glove skateboard surfboard tennis_racket bottle wine_glass cup fork knife spoon bowl banana apple sandwich orange broccoli carrot hot_dog pizza donut cake chair couch potted_plant bed dining_table toilet tv laptop mouse remote keyboard cell_phone microwave oven toaster sink refrigerator book clock vase scissors teddy_bear hair_drier toothbrush".split()
]
if len(NAMES) != 80:
    raise ValueError("expected the 80 COCO class names")


def build_yolo_layout(target="coco128"):
    """Copy the raw COCO128 files into the folder layout DataCopilot's YOLO reader expects."""
    target = Path(target)
    source = raw / "coco128"
    photos = sorted((source / "images/train2017").glob("*.jpg"))
    for number, photo in enumerate(photos):
        split = "val" if number % 5 == 4 else "train"
        (target / "images" / split).mkdir(parents=True, exist_ok=True)
        (target / "labels" / split).mkdir(parents=True, exist_ok=True)
        shutil.copy(photo, target / "images" / split / photo.name)
        label = source / "labels/train2017" / (photo.stem + ".txt")
        if label.exists():
            shutil.copy(label, target / "labels" / split / label.name)
    names = ", ".join(json.dumps(n) for n in NAMES)
    (target / "data.yaml").write_text(
        f"path: .\ntrain: images/train\nval: images/val\nnc: 80\nnames: [{names}]\n"
    )
    return target


build_yolo_layout("coco128")
healthy = dc.load("coco128", format="yolo")
for fmt, folder in [
    ("coco", "base_coco"),
    ("cvat_xml", "base_cvat"),
    ("label_studio", "base_label_studio"),
]:
    dc.export(healthy, fmt, folder)
print(
    "photos:",
    healthy.manifest.image_count,
    "| boxes:",
    healthy.manifest.annotation_count,
    "| classes used:",
    len({a.category_id for a in healthy.annotations}),
)
print(
    "starting points:",
    sorted(p.name for p in Path(".").iterdir() if p.name.startswith(("coco128", "base_"))),
)
base = datacopilot("ingest", "coco128", "--format", "yolo", "-o", "base.dcp")

photos: 128 | boxes: 929 | classes used: 71
starting points: ['base_coco', 'base_cvat', 'base_label_studio', 'coco128']


$ datacopilot ingest coco128 --format yolo -o base.dcp
d9105ffb854e7dbeccf255a4848bd692
[exit code 0]


**What this shows.** The healthy dataset ingests with exit code 0 and gets a version id. All the cases below start from copies of these folders.

## Case 1. What could go wrong: a photo is a zero-byte file

An interrupted download or copy can leave an image file with no content at all.

In [4]:
case = copy_of("coco128", "case_zero_byte")
victim = sorted((case / "images/train").glob("*.jpg"))[3]
victim.write_bytes(b"")
print(victim.name, "now has", victim.stat().st_size, "bytes")

000000000034.jpg now has 0 bytes


In [5]:
done = datacopilot("ingest", "case_zero_byte", "--format", "yolo", "-o", "case_zero_byte.dcp")
record(
    "1 zero-byte image",
    "ingest refuses (exit 1) and names the file",
    done.returncode == 1 and victim.name in said(done),
    f"exit {done.returncode}; {said(done)}",
)

$ datacopilot ingest case_zero_byte --format yolo -o case_zero_byte.dcp
[message] error: 000000000034.jpg: unsupported or invalid image format
[exit code 1]


**What this shows.** DataCopilot refuses the whole dataset (exit 1) and names the broken file, instead of silently skipping it and training on 127 photos. Nothing is written.

## Case 2. What could go wrong: a JPEG is cut off halfway

A half-copied JPEG keeps its first bytes, so its header (the part that says "I am a 640 by 480 picture") still looks fine.

In [6]:
case = copy_of("coco128", "case_truncated")
victim = sorted((case / "images/train").glob("*.jpg"))[3]
data = victim.read_bytes()
victim.write_bytes(data[: len(data) // 2])
print(victim.name, "was", len(data), "bytes and is now", victim.stat().st_size)

000000000034.jpg was 105097 bytes and is now 52548


In [7]:
ingest = datacopilot("ingest", "case_truncated", "--format", "yolo", "-o", "case_truncated.dcp")
print()
plain = datacopilot("audit", "case_truncated.dcp", "--no-hash-cache")
cut_off = [f for f in latest_report()["findings"] if f["payload"]["rule"] == "truncated_file"]
for f in cut_off:
    print("finding:", f["check"], f["severity"], f["payload"]["rule"], f["payload"]["file_name"])
print()
with_model = datacopilot(
    "audit",
    "case_truncated.dcp",
    "--no-hash-cache",
    "--embedder",
    "dinov2_small",
    "--fail-on",
    "leakage",
    timed=True,
)
for f in latest_report()["findings"]:
    if f["check"] == "leakage" and f["payload"]["rule"] == "images_not_embedded":
        print("finding:", f["check"], f["severity"], f["payload"]["rule"], f["payload"]["reasons"])
model_flagged = "images_not_embedded" in json.dumps(latest_report()["findings"])
strict_gate = datacopilot("audit", "case_truncated.dcp", "--no-hash-cache", "--fail-on", "warn")
record(
    "2 truncated JPEG",
    "the plain audit warns truncated_file (a gate on warn fails, exit 1); the model audit fails closed (exit 1)",
    with_model.returncode == 1
    and model_flagged
    and len(cut_off) == 1
    and cut_off[0]["severity"] == "WARN"
    and strict_gate.returncode == 1,
    f"ingest exit {ingest.returncode}; plain audit exit {plain.returncode} with {len(cut_off)} truncated_file warning; gate on warn exit {strict_gate.returncode}; model audit exit {with_model.returncode}",
)

$ datacopilot ingest case_truncated --format yolo -o case_truncated.dcp
63739b2b73de57a0600cdfff66e82c73
[exit code 0]



$ datacopilot audit case_truncated.dcp --no-hash-cache
passed critical=0 warn=13 info=1 run_id=a7ac3d36-4dc3-477a-b453-5558135c9366
[exit code 0]
finding: integrity WARN truncated_file 000000000034.jpg



$ datacopilot audit case_truncated.dcp --no-hash-cache --embedder dinov2_small --fail-on leakage
failed critical=1 warn=15 info=3 run_id=d793f722-089c-4a20-ab40-a07a1e259ddf
[exit code 1, 6.4 s]
finding: leakage CRITICAL images_not_embedded {'decode_failed': 1}


$ datacopilot audit case_truncated.dcp --no-hash-cache --fail-on warn
failed critical=0 warn=13 info=1 run_id=86f9adb4-1e7e-401a-8d8f-dc855adbb847
[exit code 1]


**What this shows.** Ingest reads only the header, so it accepts the file (exit 0). The plain audit adds one cheap check, a look at the last few kilobytes for the marker that ends a complete JPEG, PNG or GIF, and reports a `truncated_file`
**warning** naming the photo (exit 0 by default, exit 1 when the gate is `--fail-on warn`). The moment a model has to decode the pixels, the photo cannot be read at all, and the audit **fails closed**: exit 1 and a finding that says one image could not be
embedded. A photo the model could not look at is never silently counted as "no leak".

## Case 3. What could go wrong: an image file is really a web page

A failed download often saves the server's error page under the image's name.

In [8]:
case = copy_of("coco128", "case_html")
victim = sorted((case / "images/train").glob("*.jpg"))[3]
victim.write_text("<html><body><h1>403 Forbidden</h1></body></html>")
print(victim.name, "starts with:", victim.read_bytes()[:30])

000000000034.jpg starts with: b'<html><body><h1>403 Forbidden<'


In [9]:
done = datacopilot("ingest", "case_html", "--format", "yolo", "-o", "case_html.dcp")
record(
    "3 HTML saved as .jpg",
    "ingest refuses (exit 1) and names the file",
    done.returncode == 1 and victim.name in said(done),
    f"exit {done.returncode}; {said(done)}",
)

$ datacopilot ingest case_html --format yolo -o case_html.dcp
[message] error: 000000000034.jpg: unsupported or invalid image format
[exit code 1]


**What this shows.** DataCopilot looks at the file's real content, not its name: a web page is not an image, so the dataset is refused (exit 1) with the file named.

## Case 4. What could go wrong: the label file mentions a photo that is missing

COCO files list their photos by name. Here one listed photo has been deleted.

In [10]:
case = copy_of("base_coco", "case_missing")
(case / "images/000000000009.jpg").unlink()
print(
    "deleted images/000000000009.jpg; it is still listed in",
    [p.name for p in sorted((case / "annotations").glob("*.json"))],
)

deleted images/000000000009.jpg; it is still listed in ['instances_train.json', 'instances_val.json']


In [11]:
normal = datacopilot("ingest", "case_missing", "--format", "coco", "-o", "case_missing.dcp")
print()
strict = datacopilot(
    "ingest", "case_missing", "--format", "coco", "-o", "case_missing_strict.dcp", "--strict"
)
record(
    "4 missing image file",
    "plain ingest goes on but counts it; --strict fails (exit 1)",
    normal.returncode == 0
    and strict.returncode == 1
    and "images.file_missing=1" in said(strict, 300),
    f"plain exit {normal.returncode}; strict exit {strict.returncode}: {said(strict, 80)}",
)

$ datacopilot ingest case_missing --format coco -o case_missing.dcp
70d63cb231128257cfd574ee71a74a51
[exit code 0]



$ datacopilot ingest case_missing --format coco -o case_missing_strict.dcp --strict
[message] error: conversion coco -> dcp is lossy: annotations.of_missing_image=8, images.file_missing=1
[exit code 1]


**What this shows.** Without `--strict` DataCopilot still builds the dataset from the photos it has and counts the loss; with `--strict` the same loss
(one missing photo and the 8 boxes that belonged to it) stops the run with exit 1. CI jobs should use `--strict`.

## Case 5. What could go wrong: a label file points outside the dataset folder

A hostile or careless label file can name a photo with `..`, with an absolute path, or through a symbolic link, hoping the tool opens a file elsewhere on your computer. Here the outside
file is a real photo, and we **lock it** (permission 000) so that any attempt to open it would show up as a "permission denied" error.

In [12]:
secret = Path("secret.jpg")
shutil.copy("coco128/images/train/000000000009.jpg", secret)
secret.chmod(0)  # nobody may read it: if DataCopilot tried, we would see a permission error

dotdot = copy_of("base_coco", "case_dotdot")
edit_json(
    dotdot / "annotations/instances_train.json",
    lambda d: d["images"][0].__setitem__("file_name", "../secret.jpg"),
)

absolute = copy_of("base_coco", "case_absolute")
edit_json(
    absolute / "annotations/instances_train.json",
    lambda d: d["images"][0].__setitem__("file_name", str(secret.resolve())),
)

linked = copy_of("base_coco", "case_symlink")
(linked / "images/000000000009.jpg").unlink()
(linked / "images/000000000009.jpg").symlink_to(secret.resolve())
print("outside file:", secret, "| permissions:", oct(secret.stat().st_mode & 0o777))

outside file: secret.jpg | permissions: 0o0


In [13]:
results = {}
for label, folder in [
    ("a  '..' in the file name", "case_dotdot"),
    ("b  absolute path", "case_absolute"),
    ("c  symbolic link", "case_symlink"),
]:
    print("--", label)
    results[label[0]] = datacopilot("ingest", folder, "--format", "coco", "-o", folder + ".dcp")
for key, name in [
    ("a", "5a '..' in a file name"),
    ("b", "5b absolute path"),
    ("c", "5c symbolic link"),
]:
    done = results[key]
    record(
        name,
        "refused (exit 1) as an unsafe path, never opened",
        done.returncode == 1 and "unsafe path" in said(done) and "ermission" not in said(done),
        f"exit {done.returncode}; {said(done, 90)}",
    )
secret.chmod(0o600)

-- a  '..' in the file name


$ datacopilot ingest case_dotdot --format coco -o case_dotdot.dcp
[message] error: unsafe path '../secret.jpg': must be relative and stay inside the dataset
[exit code 1]
-- b  absolute path


$ datacopilot ingest case_absolute --format coco -o case_absolute.dcp
[message] error: unsafe path '<work>/secret.jpg': must be relative and stay inside the dataset
[exit code 1]
-- c  symbolic link


$ datacopilot ingest case_symlink --format coco -o case_symlink.dcp
[message] error: unsafe path '000000000009.jpg': resolves outside the dataset
[exit code 1]


**What this shows.** All three attempts are refused with exit 1 and the words "unsafe path". Because the outside file was locked and the error is *not* "permission denied", DataCopilot rejected the path by its name
before trying to open anything.

### Case 5d. What could go wrong: the dataset folder is moved away from its photos

A `.dcp` folder remembers where the photos are (it holds tables of images, boxes and classes plus `manifest.json`, a small description file with the version id; the photos are not inside). If you copy it somewhere else, the photos it points to are now outside its neighbourhood, and DataCopilot will not read files from there unless you say so.

In [14]:
Path("elsewhere").mkdir(exist_ok=True)
shutil.copytree("base.dcp", "elsewhere/moved.dcp", symlinks=True)
print("moved copy:", sorted(p.name for p in Path("elsewhere/moved.dcp").iterdir()))

moved copy: ['annotations.parquet', 'categories.parquet', 'images.parquet', 'manifest.json', 'versions']


In [15]:
done = datacopilot("audit", "elsewhere/moved.dcp", "--no-hash-cache")
rules = Counter(
    f["payload"]["rule"] for f in latest_report()["findings"] if f["severity"] == "CRITICAL"
)
print("critical findings:", dict(rules))
record(
    "5d dataset moved from its photos",
    "audit fails closed (exit 1): photos not examined",
    done.returncode == 1 and "external_not_checked" in rules,
    f"exit {done.returncode}; critical: {dict(rules)}",
)

$ datacopilot audit elsewhere/moved.dcp --no-hash-cache
failed critical=2 warn=12 info=1 run_id=d89c659d-a741-493d-af12-364bfe954a9a
[exit code 1]
critical findings: {'external_not_checked': 1, 'images_not_hashed': 1}


**What this shows.** The audit fails (exit 1) instead of passing without having looked at any photo: "not examined" is treated as a critical finding. You can allow outside photos on purpose with `--allow-external-images`.

## Case 6. What could go wrong: a box sticks out of its photo

The first box of a COCO file is changed so that it is 5000 pixels wide on a 640-pixel photo.

In [16]:
case = copy_of("base_coco", "case_box_outside")
edit_json(
    case / "annotations/instances_train.json",
    lambda d: d["annotations"][0].__setitem__("bbox", [10, 10, 5000, 50]),
)
print(
    "first box of instances_train.json is now [x, y, width, height] =",
    json.loads((case / "annotations/instances_train.json").read_text())["annotations"][0]["bbox"],
)

first box of instances_train.json is now [x, y, width, height] = [10, 10, 5000, 50]


In [17]:
done = datacopilot("ingest", "case_box_outside", "--format", "coco", "-o", "case_box_outside.dcp")
record(
    "6 box outside the image",
    "ingest refuses (exit 1) naming the box",
    done.returncode == 1 and "outside image" in said(done),
    f"exit {done.returncode}; {said(done, 90)}",
)

$ datacopilot ingest case_box_outside --format coco -o case_box_outside.dcp
[message] error: invalid dataset: ann_id 'instances_train:1' box (10.0, 10.0, 5010.0, 60.0) outside image '3b0bb6bebad60a33' (640x480)
[exit code 1]


**What this shows.** A box that does not fit inside its photo is a broken label, so the dataset is not created (exit 1). The message names the box and the photo size so you can find it.

## Case 7. What could go wrong: a box has zero area

A box that is 0 pixels wide is a label that points at nothing.

In [18]:
case = copy_of("base_coco", "case_zero_area")
edit_json(
    case / "annotations/instances_train.json",
    lambda d: d["annotations"][0].__setitem__("bbox", [10, 10, 0, 50]),
)
print(
    "first box of instances_train.json is now [x, y, width, height] =",
    json.loads((case / "annotations/instances_train.json").read_text())["annotations"][0]["bbox"],
)

first box of instances_train.json is now [x, y, width, height] = [10, 10, 0, 50]


In [19]:
done = datacopilot("ingest", "case_zero_area", "--format", "coco", "-o", "case_zero_area.dcp")
record(
    "7 zero-area box",
    "ingest refuses (exit 1)",
    done.returncode == 1 and "box must satisfy" in said(done),
    f"exit {done.returncode}; {said(done, 90)}",
)

$ datacopilot ingest case_zero_area --format coco -o case_zero_area.dcp
[message] error: annotation 1: row: Value error, box must satisfy 0 <= x_min < x_max and 0 <= y_min < y_max
[exit code 1]


**What this shows.** An empty box is refused when the dataset is read (exit 1), so it can never reach training or even the audit.

## Case 8. What could go wrong: a coordinate is "not a number" (NaN)

Some tools write `NaN` into JSON when a calculation fails. Standard JSON has no such value, and a box with an unknown corner is meaningless.

In [20]:
case = copy_of("base_coco", "case_nan")
edit_json(
    case / "annotations/instances_train.json",
    lambda d: d["annotations"][0].__setitem__("bbox", [1.0, 2.0, float("nan"), 4.0]),
)
print(
    "what is in the file:",
    re.search(
        r'"bbox": \[1.0, 2.0, [^,]*, 4.0\]', (case / "annotations/instances_train.json").read_text()
    ).group(0),
)

what is in the file: "bbox": [1.0, 2.0, NaN, 4.0]


In [21]:
done = datacopilot("ingest", "case_nan", "--format", "coco", "-o", "case_nan.dcp")
record(
    "8 NaN coordinate in COCO JSON",
    "ingest refuses (exit 1) as invalid JSON",
    done.returncode == 1 and "not valid JSON" in said(done),
    f"exit {done.returncode}; {said(done, 90)}",
)

$ datacopilot ingest case_nan --format coco -o case_nan.dcp
[message] error: instances_train.json is not valid JSON
[exit code 1]


**What this shows.** The file is rejected as invalid JSON (exit 1), so a NaN can never slip into a calculation later.

## Case 9. What could go wrong: the same box twice, and one box with two classes

Labelling tools sometimes save a box twice, or two people label the same object with different classes.

In [22]:
case = copy_of("coco128", "case_duplicates")
label = sorted((case / "labels/train").glob("*.txt"))[0]
lines = label.read_text().splitlines()
first_class, *box = lines[0].split()
label.write_text(
    "\n".join(lines + [lines[0], f"{(int(first_class) + 1) % 80} " + " ".join(box)]) + "\n"
)
print(
    label.name,
    "had",
    len(lines),
    "boxes; we added an exact copy of the first box and the same box with the next class",
)

000000000009.txt had 8 boxes; we added an exact copy of the first box and the same box with the next class


In [23]:
ingest = datacopilot("ingest", "case_duplicates", "--format", "yolo", "-o", "case_duplicates.dcp")
done = datacopilot("audit", "case_duplicates.dcp", "--no-hash-cache", "--fail-on", "warn")
hits = [
    f
    for f in latest_report()["findings"]
    if f["payload"]["rule"] in ("duplicate_box", "conflicting_class")
    and f["payload"].get("iou", 0) >= 0.999
]
for f in hits:
    print("finding:", f["payload"]["rule"], "| overlap", f["payload"]["iou"])
record(
    "9 duplicate box + two classes on one box",
    "warnings duplicate_box and conflicting_class; gate on warn fails (exit 1)",
    done.returncode == 1
    and {f["payload"]["rule"] for f in hits} == {"duplicate_box", "conflicting_class"},
    f"exit {done.returncode}; rules {sorted({f['payload']['rule'] for f in hits})}",
)

$ datacopilot ingest case_duplicates --format yolo -o case_duplicates.dcp
26879afd2e98a2572cf6ddabeee1afde
[exit code 0]


$ datacopilot audit case_duplicates.dcp --no-hash-cache --fail-on warn
failed critical=0 warn=14 info=1 run_id=b1c1eec5-663d-4982-87e0-934950279b68
[exit code 1]
finding: duplicate_box | overlap 1.0
finding: conflicting_class | overlap 1.0


**What this shows.** Both problems are caught as warnings with an overlap of 1.0 (identical rectangles). They are warnings, not critical, so the default gate would let them pass, but a stricter gate
(`--fail-on warn`) stops the build (exit 1).

## Case 10. What could go wrong: a class name that is 5,000 characters long

Class names are data, and data can be hostile or just wrong. A huge name could flood a terminal or a report.

In [24]:
case = copy_of("base_coco", "case_long_name")
for split in ("train", "val"):
    edit_json(
        case / f"annotations/instances_{split}.json",
        lambda d: d["categories"][0].__setitem__("name", "x" * 5000),
    )
print("class 0 is now called x * 5000; name length:", 5000)

class 0 is now called x * 5000; name length: 5000


In [25]:
ingest = datacopilot("ingest", "case_long_name", "--format", "coco", "-o", "case_long_name.dcp")
done = datacopilot("audit", "case_long_name.dcp", "--no-hash-cache")
longest = {}
for report_file in ("report.html", "report.json", "report.sarif"):
    runs = re.findall(r"x{20,}", Path(report_file).read_text())
    longest[report_file] = max((len(r) for r in runs), default=0)
print("longest run of x in each report:", longest)
record(
    "10 5000-character class name",
    "accepted; every report cuts the name to the documented 100 or 200 characters",
    ingest.returncode == 0 and done.returncode == 0 and max(longest.values()) <= 200,
    f"exit {ingest.returncode}/{done.returncode}; longest name shown: {max(longest.values())} characters",
)

$ datacopilot ingest case_long_name --format coco -o case_long_name.dcp
bf3d4ab29b0fb3c4dfbe9ee48c556e6a
[exit code 0]


$ datacopilot audit case_long_name.dcp --no-hash-cache
passed critical=0 warn=12 info=1 run_id=15c956f9-e521-4719-aa8d-5adb9b08347d
[exit code 0]
longest run of x in each report: {'report.html': 99, 'report.json': 200, 'report.sarif': 99}


**What this shows.** The dataset is valid, so it is accepted (exit 0), but none of the three reports (`report.html` to read, `report.json` for programs, `report.sarif` for code-scanning tools) shows more than 200 characters of the name: the web page and the code-scanning file cut it to 99, the JSON file keeps up to 200 (DataCopilot documents "100 or 200"). Odd data cannot blow up your CI logs.

## Case 11. What could go wrong: the dataset has no images at all

An empty dataset passes every check trivially ("no broken boxes!"), which would be dangerous. DataCopilot must fail closed.

In [26]:
case = Path("case_empty/annotations")
case.mkdir(parents=True)
(case / "instances_train.json").write_text(
    json.dumps({"images": [], "annotations": [], "categories": [{"id": 1, "name": "thing"}]})
)

from datetime import datetime, timezone

nothing = dc.Dataset(
    dc.DatasetManifest(
        version="0" * 32,
        schema_version=1,
        name="empty",
        task="detection",
        categories=(),
        image_count=0,
        annotation_count=0,
        created_at=datetime(2026, 1, 1, tzinfo=timezone.utc),
    ),
    (),
    (),
)
print("an in-memory dataset with", nothing.manifest.image_count, "images")

an in-memory dataset with 0 images


In [27]:
done = datacopilot("ingest", "case_empty", "--format", "coco", "-o", "case_empty.dcp")
try:
    dc.audit(nothing)
    api = "no error"
except dc.DataCopilotError as problem:
    api = f"{type(problem).__name__}: {problem}"
print("dc.audit on the empty dataset ->", api)
record(
    "11 empty dataset",
    "ingest refuses (exit 1); the Python audit raises an error",
    done.returncode == 1 and "no images" in said(done) and api.startswith("ValidationError"),
    f"exit {done.returncode}; {said(done, 50)}; API {api[:40]}",
)

$ datacopilot ingest case_empty --format coco -o case_empty.dcp
[message] error: no images found in the input
[exit code 1]
dc.audit on the empty dataset -> ValidationError: dataset has no images; there is nothing to audit


**What this shows.** Both ways in refuse an empty dataset: the command exits 1 with "no images found", and the Python audit raises a clear error rather than returning an empty, passing report.

## Case 12. What could go wrong: the same picture under two names

Two files with identical bytes are the same photo. In one split it only wastes space; across splits it is leakage.

In [28]:
same_split = copy_of("coco128", "case_same_split")
original = sorted((same_split / "images/train").glob("*.jpg"))[0]
shutil.copy(original, original.with_name("copy_of_first.jpg"))
shutil.copy(
    same_split / "labels/train" / (original.stem + ".txt"),
    same_split / "labels/train/copy_of_first.txt",
)

across = copy_of("coco128", "case_across_splits")
shutil.copy(across / "images/train" / original.name, across / "images/val/copy_across.jpg")
shutil.copy(
    across / "labels/train" / (original.stem + ".txt"), across / "labels/val/copy_across.txt"
)
print(
    "copies of",
    original.name,
    ": one in train (copy_of_first.jpg) and one in val (copy_across.jpg), in two separate datasets",
)

copies of 000000000009.jpg : one in train (copy_of_first.jpg) and one in val (copy_across.jpg), in two separate datasets


In [29]:
a = datacopilot("ingest", "case_same_split", "--format", "yolo", "-o", "case_same_split.dcp")
c = datacopilot("ingest", "case_across_splits", "--format", "yolo", "-o", "case_across_splits.dcp")
record(
    "12a same content twice in one split",
    "ingest refuses (exit 1) naming both files",
    a.returncode == 1
    and "identical content" in said(a, 300)
    and "copy_of_first.jpg" in said(a, 300),
    f"exit {a.returncode}; {said(a, 70)}",
)
record(
    "12b same content across splits",
    "ingest refuses (exit 1) naming both files",
    c.returncode == 1 and "identical content" in said(c, 300) and "copy_across.jpg" in said(c, 300),
    f"exit {c.returncode}; {said(c, 70)}",
)

$ datacopilot ingest case_same_split --format yolo -o case_same_split.dcp
[message] error: images '000000000009.jpg' and 'copy_of_first.jpg' have identical content (image_id 3b0bb6bebad60a33); remove or deduplicate one before ingesting
[exit code 1]


$ datacopilot ingest case_across_splits --format yolo -o case_across_splits.dcp
[message] error: images '000000000009.jpg' and 'copy_across.jpg' have identical content (image_id 3b0bb6bebad60a33); remove or deduplicate one before ingesting
[exit code 1]


**What this shows.** Identical files are refused right away (exit 1) with both file names, in the same split or across splits. Exact copies are the easy case; *re-saved* copies, whose bytes
differ, are the hard case, and the first notebook showed a model catching those.

## Case 13. What could go wrong: a photo changes after you ingested the dataset

A `.dcp` folder points at the photos; it does not copy them. If somebody edits a photo afterwards, the data you audit is no longer the data you recorded.

In [30]:
case = copy_of("coco128", "case_changed")
ingest = datacopilot("ingest", "case_changed", "--format", "yolo", "-o", "case_changed.dcp")
victim = sorted((case / "images/train").glob("*.jpg"))[5]
with PILImage.open(victim) as photo:
    photo.save(victim, quality=50)  # same picture, new bytes
print(victim.name, "was re-saved after the ingest")

$ datacopilot ingest case_changed --format yolo -o case_changed.dcp
d9105ffb854e7dbeccf255a4848bd692
[exit code 0]
000000000049.jpg was re-saved after the ingest


In [31]:
done = datacopilot("audit", "case_changed.dcp", "--no-hash-cache")
changed = [f for f in latest_report()["findings"] if f["payload"]["rule"] == "content_changed"]
for f in changed:
    print("finding:", f["severity"], f["check"], f["payload"]["rule"], f["payload"]["file_name"])
record(
    "13 file changed after ingest",
    "CRITICAL content_changed names the file; gate fails (exit 1)",
    done.returncode == 1
    and len(changed) == 1
    and changed[0]["severity"] == "CRITICAL"
    and changed[0]["payload"]["file_name"] == victim.name,
    f"exit {done.returncode}; {len(changed)} CRITICAL content_changed ({victim.name})",
)

$ datacopilot audit case_changed.dcp --no-hash-cache
failed critical=1 warn=12 info=1 run_id=08cc622f-b9ed-48b1-9c59-b4f1ccf40d14
[exit code 1]
finding: CRITICAL duplicates content_changed 000000000049.jpg


**What this shows.** The audit notices that the file no longer matches the fingerprint taken at ingest time, reports it as **critical** and names the file, so the default gate fails (exit 1).

## Case 14. What could go wrong: images the model cannot read

Two kinds of picture are hard for an image model: formats it does not support (WebP here) and pictures so large that decoding them could use gigabytes of memory (a "decompression bomb").

In [32]:
# (a) a WebP photo dropped into a YOLO dataset
webp = copy_of("coco128", "case_webp")
with PILImage.open(sorted((webp / "images/train").glob("*.jpg"))[0]) as photo:
    photo.save(webp / "images/train/extra_photo.webp")

# (b) a real, valid PNG that is 10,000 x 10,000 pixels (100 megapixels) but only a few kilobytes big
import struct
import zlib


def huge_png(path, side=10_000):
    def chunk(kind, payload):
        body = kind + payload
        return struct.pack(">I", len(payload)) + body + struct.pack(">I", zlib.crc32(body))

    packer = zlib.compressobj(9)
    row = bytes((side + 7) // 8 + 1)  # one all-black row of 1-bit pixels
    pixels = b"".join(packer.compress(row) for _ in range(side)) + packer.flush()
    path.write_bytes(
        b"\x89PNG\r\n\x1a\n"
        + chunk(b"IHDR", struct.pack(">IIBBBBB", side, side, 1, 0, 0, 0, 0))
        + chunk(b"IDAT", pixels)
        + chunk(b"IEND", b"")
    )


bomb = copy_of("base_coco", "case_huge_png")
huge_png(bomb / "images/huge.png")


def add_photo(d):
    d["images"].append({"id": 99999, "file_name": "huge.png", "width": 10_000, "height": 10_000})
    d["annotations"].append(
        {"id": 99999, "image_id": 99999, "category_id": 0, "bbox": [10, 10, 50, 50], "iscrowd": 0}
    )


edit_json(bomb / "annotations/instances_val.json", add_photo)
print(
    "huge.png:",
    (bomb / "images/huge.png").stat().st_size,
    "bytes on disk,",
    10_000 * 10_000 // 1_000_000,
    "megapixels",
)

huge.png: 12227 bytes on disk, 100 megapixels


In [33]:
print("-- (a) WebP in a YOLO dataset")
w1 = datacopilot("ingest", "case_webp", "--format", "yolo", "-o", "case_webp.dcp")
w2 = datacopilot(
    "ingest", "case_webp", "--format", "yolo", "-o", "case_webp_strict.dcp", "--strict"
)
print(
    "did anything get written?",
    Path("case_webp.dcp").exists() or Path("case_webp_strict.dcp").exists(),
)
print()
print("-- (b) 100-megapixel PNG in a COCO dataset")
p1 = datacopilot("ingest", "case_huge_png", "--format", "coco", "-o", "case_huge_png.dcp")
p2 = datacopilot("audit", "case_huge_png.dcp", "--no-hash-cache")
p3 = datacopilot(
    "audit",
    "case_huge_png.dcp",
    "--no-hash-cache",
    "--embedder",
    "dinov2_small",
    "--fail-on",
    "leakage",
    timed=True,
)
unembedded = [
    f
    for f in latest_report()["findings"]
    if f["check"] == "leakage" and f["payload"]["rule"] == "images_not_embedded"
]
print("finding:", [(f["severity"], f["payload"]["reasons"]) for f in unembedded])
record(
    "14a WebP image",
    "refused (exit 1) naming the file, never skipped; nothing written",
    w1.returncode == 1
    and w2.returncode == 1
    and "extra_photo.webp: unsupported image format" in said(w1)
    and not Path("case_webp.dcp").exists(),
    f"exit {w1.returncode}; {said(w1, 90)}",
)
record(
    "14b 100-megapixel PNG",
    "model audit fails the gate (exit 1) with a not-embedded finding",
    p3.returncode == 1 and len(unembedded) == 1 and unembedded[0]["severity"] == "CRITICAL",
    f"ingest {p1.returncode}, plain audit {p2.returncode}, model audit exit {p3.returncode}; reason {unembedded[0]['payload']['reasons'] if unembedded else None}",
)

-- (a) WebP in a YOLO dataset


$ datacopilot ingest case_webp --format yolo -o case_webp.dcp
[message] error: extra_photo.webp: unsupported image format (.webp); convert it to JPEG, PNG, BMP or GIF
[exit code 1]


$ datacopilot ingest case_webp --format yolo -o case_webp_strict.dcp --strict
[message] error: extra_photo.webp: unsupported image format (.webp); convert it to JPEG, PNG, BMP or GIF
[exit code 1]
did anything get written? False

-- (b) 100-megapixel PNG in a COCO dataset


$ datacopilot ingest case_huge_png --format coco -o case_huge_png.dcp
a02dbb8750af67de9786c9b4a483d96b
[exit code 0]


$ datacopilot audit case_huge_png.dcp --no-hash-cache
passed critical=0 warn=13 info=1 run_id=96b9798a-c133-4119-b2fa-ab582a9a256a
[exit code 0]


$ datacopilot audit case_huge_png.dcp --no-hash-cache --embedder dinov2_small --fail-on leakage
failed critical=1 warn=15 info=3 run_id=4a9a6d3b-dbf8-4c2e-93af-6aab0e4fae79
[exit code 1, 6.1 s]
finding: [('CRITICAL', {'too_many_pixels': 1})]


**What this shows.** (a) DataCopilot cannot identify WebP pictures, and no reader skips a picture silently any more: ingest stops (exit 1, with or without `--strict`), names the file and writes nothing, so the photo cannot vanish from the audit. (b) The
huge PNG is a valid file that ingest and the plain audit accept, but the model refuses to decode a 100-megapixel picture, and the audit reports a **critical** "not embedded" finding (reason `too_many_pixels`) so the leakage gate fails (exit 1). Nothing
is skipped silently.

## Case 15. What could go wrong: a CVAT file hides an "entity bomb"

XML can define abbreviations ("entities") that expand into other abbreviations. Ten levels of ten copies is a billion words from a few hundred bytes: the "billion laughs" attack, which freezes a naive reader.

In [34]:
laughs = (
    '<!DOCTYPE annotations [<!ENTITY l0 "lol">'
    + "".join(f'<!ENTITY l{n + 1} "' + f"&l{n};" * 10 + '">' for n in range(9))
    + "]>"
)
bomb = copy_of("base_cvat", "case_xml_bomb")
target = bomb / "annotations/val.xml"
text = target.read_text()
target.write_text(
    text.replace("<annotations>", laughs + "\n<annotations>", 1).replace(
        "<version>1.1</version>", "<version>&l9;</version>", 1
    )
)
plain = copy_of("base_cvat", "case_doctype")
target = plain / "annotations/val.xml"
target.write_text(
    target.read_text().replace("<annotations>", "<!DOCTYPE annotations>\n<annotations>", 1)
)
print(
    "the bomb adds only",
    len(laughs),
    "characters; fully expanded it would be",
    f"{3 * 10**9:,}",
    "characters",
)

the bomb adds only 538 characters; fully expanded it would be 3,000,000,000 characters


In [35]:
x1 = datacopilot(
    "ingest", "case_xml_bomb", "--format", "cvat_xml", "-o", "case_xml_bomb.dcp", timed=True
)
x2 = datacopilot(
    "ingest", "case_doctype", "--format", "cvat_xml", "-o", "case_doctype.dcp", timed=True
)
record(
    "15a XML entity bomb (CVAT)",
    "rejected quickly (exit 1, under 5 s)",
    x1.returncode == 1 and "DOCTYPE" in said(x1) and x1.seconds < 5,
    f"exit {x1.returncode} in {x1.seconds:.1f} s; {said(x1, 60)}",
)
record(
    "15b plain DOCTYPE (CVAT)",
    "rejected (exit 1)",
    x2.returncode == 1 and "DOCTYPE" in said(x2),
    f"exit {x2.returncode}; {said(x2, 70)}",
)

$ datacopilot ingest case_xml_bomb --format cvat_xml -o case_xml_bomb.dcp
[message] error: val.xml: DOCTYPE and entity declarations are not allowed in dataset XML
[exit code 1, 0.3 s]


$ datacopilot ingest case_doctype --format cvat_xml -o case_doctype.dcp
[message] error: val.xml: DOCTYPE and entity declarations are not allowed in dataset XML
[exit code 1, 0.3 s]


**What this shows.** DataCopilot never expands entities: any `DOCTYPE` or entity declaration is refused before anything is parsed, so the bomb is rejected in a fraction of a second (exit 1) and a harmless `DOCTYPE` is refused the same way.

## Case 16. What could go wrong: a Label Studio export carries private information

Label Studio exports can include who labelled each task (names and e-mail addresses), free-form notes, and temporary signed links (a "presigned URL" is a web address with a secret signature that grants access to a private file). They can also contain skipped tasks and
model predictions without a score. We add all of these to a real export.

In [36]:
case = copy_of("base_label_studio", "case_label_studio")
file = case / "annotations/val.json"
tasks = json.loads(file.read_text())
labelled = [t for t in tasks if t["annotations"] and t["annotations"][0]["result"]]

a = labelled[
    0
]  # who labelled it, a private note, and a signed link in a field that is not the image
a["annotations"][0]["completed_by"] = {"id": 7, "email": "annotator.one@example.org"}
a["annotations"][0]["created_username"] = "annotator.one@example.org, 7"
a["meta"] = {"reviewer": "boss@example.org"}
a["data"]["source_url"] = (
    "https://bucket.s3.amazonaws.com/restricted/x.jpg?X-Amz-Signature=abc123deadbeef&X-Amz-Credential=EXAMPLE-KEY-ID"
)

labelled[1]["annotations"][0]["was_cancelled"] = True  # the annotator pressed "skip" on this task

c = labelled[2]  # a model prediction whose score is null
first = c["annotations"][0]["result"][0]
c["predictions"] = [
    {
        "model_version": "demo",
        "score": None,
        "result": [
            {
                "original_width": first["original_width"],
                "original_height": first["original_height"],
                "from_name": "label",
                "to_name": "image",
                "type": "rectanglelabels",
                "value": {
                    "x": 10,
                    "y": 10,
                    "width": 20,
                    "height": 20,
                    "rotation": 0,
                    "rectanglelabels": ["person"],
                },
            }
        ],
    }
]

d = labelled[3]  # a task whose image is a signed web link
d["data"]["image"] = "https://bucket.s3.amazonaws.com/photos/y.jpg?X-Amz-Signature=feedface0123"
file.write_text(json.dumps(tasks))
print("tasks in val.json:", len(tasks), "| edited 4 of them")

tasks in val.json: 25 | edited 4 of them


In [37]:
normal = datacopilot(
    "ingest", "case_label_studio", "--format", "label_studio", "-o", "case_label_studio.dcp"
)
strict = datacopilot(
    "ingest",
    "case_label_studio",
    "--format",
    "label_studio",
    "-o",
    "case_label_studio_strict.dcp",
    "--strict",
)
audit = datacopilot("audit", "case_label_studio.dcp", "--no-hash-cache")
export = datacopilot(
    "export", "case_label_studio.dcp", "--format", "label_studio", "-o", "case_label_studio_out"
)

$ datacopilot ingest case_label_studio --format label_studio -o case_label_studio.dcp
5428a2a7e90860707dc38c134f436371
[exit code 0]


$ datacopilot ingest case_label_studio --format label_studio -o case_label_studio_strict.dcp --strict
[message] error: conversion label_studio -> dcp is lossy: images.source.annotation.completed_by=1, images.source.annotation.created_username=1, images.source.data.source_url=1, images.source.meta=1, annotations.cancelled=16, annotations.of_missing_image=2, images.remote_uri=1
[exit code 1]


$ datacopilot audit case_label_studio.dcp --no-hash-cache
passed critical=0 warn=12 info=0 run_id=2feb3a5f-7392-4f80-b2ae-16c9b2619a72
[message] note: 1 model-generated annotation (provenance: model) excluded from every check; pass --include-model-annotations to audit them
[exit code 0]


$ datacopilot export case_label_studio.dcp --format label_studio -o case_label_studio_out
case_label_studio_out
[exit code 0]


In [38]:
secrets = [
    "example.org",
    "X-Amz",
    "EXAMPLE-KEY",
    "abc123deadbeef",
    "feedface0123",
    "annotator.one",
    "boss@",
]


def scan(*places):
    """How many files (raw bytes) under these places contain each secret."""
    hits = Counter()
    for place in places:
        for file in (
            [Path(place)]
            if Path(place).is_file()
            else [p for p in Path(place).rglob("*") if p.is_file()]
        ):
            blob = file.read_bytes()
            for secret_text in secrets:
                hits[secret_text] += secret_text.encode() in blob
    return hits


stored = dc.load("case_label_studio.dcp")
in_memory = repr(stored.images) + repr(stored.annotations) + repr(stored.manifest)
found_files = scan(
    "case_label_studio.dcp",
    "report.html",
    "report.json",
    "report.sarif",
    "case_label_studio_out/annotations",
)
found_memory = [s for s in secrets if s in in_memory]
print(
    "secrets found in the stored dataset, reports or re-exported file:", sum(found_files.values())
)
print("secrets found in the loaded tables:", len(found_memory))
dropped = said(strict, 600).split("lossy: ", 1)[1]
print("what ingest dropped and counted:", dropped)
record(
    "16 Label Studio privacy",
    "e-mails and signed URLs appear nowhere; the skip is counted; the null-score prediction stays out of the checks",
    normal.returncode == 0
    and sum(found_files.values()) == 0
    and not found_memory
    and "annotations.cancelled" in dropped
    and "completed_by" in dropped
    and "remote_uri" in dropped
    and any("model-generated" in m for m in audit.messages),
    f"exit {normal.returncode}; secrets found: {sum(found_files.values()) + len(found_memory)}; dropped fields named: {len(dropped.split(','))}",
)

secrets found in the stored dataset, reports or re-exported file: 0
secrets found in the loaded tables: 0
what ingest dropped and counted: images.source.annotation.completed_by=1, images.source.annotation.created_username=1, images.source.data.source_url=1, images.source.meta=1, annotations.cancelled=16, annotations.of_missing_image=2, images.remote_uri=1


**What this shows.** The export is read (exit 0) but only fields on a fixed allow-list are kept. None of the e-mail addresses, the signed-link signatures or the key-like text appears in the stored dataset, in any of the three reports, or in the
re-exported Label Studio file. The `--strict` run lists, by *name only*, everything that was dropped: who labelled it, the notes, the extra link field, the skipped (cancelled) annotation, and the task whose image was a web link (counted, never downloaded). The prediction with a null score was read safely and left out of the checks, as the audit's note says.

## Case 17. What could go wrong: a conversion silently loses information

Some formats have no place for some fields. Here we ingest the YOLO dataset with a *group* for each photo (the name of its folder, standing in for "which camera took it"), then convert to CVAT, which cannot store groups.

In [39]:
grouped = datacopilot(
    "ingest",
    "coco128",
    "--format",
    "yolo",
    "--group-by",
    "re:^images/(?P<g>[^/]+)/",
    "-o",
    "grouped.dcp",
)

$ datacopilot ingest coco128 --format yolo --group-by 're:^images/(?P<g>[^/]+)/' -o grouped.dcp
ac3d586ef69cf88dcc8d743b1ff63ea0
[exit code 0]


In [40]:
strict = datacopilot(
    "export", "grouped.dcp", "--format", "cvat_xml", "-o", "cvat_strict", "--strict"
)
print("folder written by the strict export:", Path("cvat_strict").exists())
lossy = dc.export(dc.load("grouped.dcp"), "cvat_xml", "cvat_lossy")
print("without --strict the loss is counted:", lossy.dropped, "| skipped:", lossy.skipped or "none")
record(
    "17 lossy conversion (YOLO to CVAT)",
    "--strict fails (exit 1) naming images.group_id and writes nothing",
    strict.returncode == 1
    and "images.group_id=128" in said(strict, 200)
    and not Path("cvat_strict").exists(),
    f"strict exit {strict.returncode}: {said(strict, 80)}; loss counts {lossy.dropped}",
)

$ datacopilot export grouped.dcp --format cvat_xml -o cvat_strict --strict
[message] error: conversion dcp -> cvat_xml is lossy: images.group_id=128
[exit code 1]
folder written by the strict export: False
without --strict the loss is counted: {'images.group_id': 128} | skipped: none


**What this shows.** With `--strict` the export stops with exit 1 and names exactly what would be lost (`images.group_id=128`) and writes nothing. Without it the same loss is counted and returned, so you can decide whether it matters.

## Case 18. What could go wrong: the output folder cannot be written

Read-only disks, wrong permissions and full disks are everyday problems. The command should say so in one clear line, not with a wall of technical text.

In [41]:
locked = Path("read_only")
locked.mkdir(exist_ok=True)
locked.chmod(0o555)  # read and enter, but not create anything
print(
    "folder permissions:",
    oct(locked.stat().st_mode & 0o777),
    "| can we write there?",
    os.access(locked, os.W_OK),
)

folder permissions: 0o555 | can we write there? False


In [42]:
done = datacopilot("ingest", "coco128", "--format", "yolo", "-o", "read_only/new.dcp")
locked.chmod(0o755)
record(
    "18 unwritable output folder",
    "exit 1 with one clear error line naming the file",
    done.returncode == 1
    and said(done) == "error: Permission denied: new.dcp"
    and len(done.messages) == 1,
    f"exit {done.returncode}; {said(done, 70)}",
)

$ datacopilot ingest coco128 --format yolo -o read_only/new.dcp
[message] error: Permission denied: new.dcp
[exit code 1]


**What this shows.** The run fails with exit 1 and a single readable `error:` line that says "Permission denied", and nothing half-written is left behind. (Finding this case in a first run of this notebook led to a small fix: the command used to exit 1 *without any readable message*.)

## Case 19. What could go wrong: a model whose licence you have not approved

Every model has a licence, and your organisation may only allow some. DataCopilot checks the licence against `DATACOPILOT_ALLOWED_LICENSES` *before* it downloads or reads anything. Here the allowed list is only `MIT`, but DINOv2-small is Apache-2.0.
To prove that nothing is downloaded, the model cache folder is pointed at an empty folder, and we look inside afterwards.

In [43]:
empty_cache = Path("empty_model_cache")
empty_cache.mkdir(exist_ok=True)
refuse_env = {
    "DATACOPILOT_ALLOWED_LICENSES": "MIT",
    "HF_HOME": str(empty_cache.resolve()),
    "HF_HUB_CACHE": str((empty_cache / "hub").resolve()),
}
print("model cache folder holds", len(list(empty_cache.rglob("*"))), "files before")

model cache folder holds 0 files before


In [44]:
refused = datacopilot(
    "audit",
    "base.dcp",
    "--no-hash-cache",
    "--embedder",
    "dinov2_small",
    "--fail-on",
    "leakage",
    env=refuse_env,
)
refused_embed = datacopilot("embed", "base.dcp", "--embedder", "dinov2_small", env=refuse_env)
print("model cache folder holds", len(list(empty_cache.rglob("*"))), "files after")
print()
allowed = datacopilot(
    "audit",
    "base.dcp",
    "--no-hash-cache",
    "--embedder",
    "dinov2_small",
    "--fail-on",
    "leakage",
    env={"DATACOPILOT_ALLOWED_LICENSES": "MIT,Apache-2.0"},
)
record(
    "19 model licence not allowed",
    "exit 1 before any download; allowed licence works (exit 0)",
    refused.returncode == 1
    and refused_embed.returncode == 1
    and "DATACOPILOT_ALLOWED_LICENSES" in said(refused, 300)
    and not list(empty_cache.rglob("*"))
    and allowed.returncode == 0,
    f"refused exit {refused.returncode}/{refused_embed.returncode}, files downloaded: {len(list(empty_cache.rglob('*')))}; with Apache-2.0 allowed exit {allowed.returncode}",
)

$ datacopilot audit base.dcp --no-hash-cache --embedder dinov2_small --fail-on leakage
[message] error: model 'dinov2_small' is licensed 'Apache-2.0', which is not in DATACOPILOT_ALLOWED_LICENSES; add it there if you have accepted its terms
[exit code 1]


$ datacopilot embed base.dcp --embedder dinov2_small
[message] error: model 'dinov2_small' is licensed 'Apache-2.0', which is not in DATACOPILOT_ALLOWED_LICENSES; add it there if you have accepted its terms
[exit code 1]
model cache folder holds 0 files after



$ datacopilot audit base.dcp --no-hash-cache --embedder dinov2_small --fail-on leakage
passed critical=0 warn=12 info=3 run_id=6596ee4b-fdf2-44b9-8b7b-c22005379213
[exit code 0]


**What this shows.** With only `MIT` allowed, both `audit` and `embed` refuse the Apache-2.0 model with exit 1 and tell you which variable to change; the empty cache folder proves that no weights were downloaded. Adding `Apache-2.0` to the list makes the same command pass (exit 0).

## Case 20. What could go wrong: a round trip quietly changes the data

You ingest a dataset, export it, and ingest the export (for example when a colleague hands the files on). If the version id changed, nobody could tell whether the data had changed or only the number formatting. Real labels make this hard: many of the real boxes touch or very slightly overshoot the photo edge, and
file formats print numbers with limited digits. Here the check runs on all 128 real photos and 929 real boxes, once for every format.

In [45]:
trips = {}
for fmt in ["yolo", "coco", "voc", "cvat_xml", "label_studio"]:
    start = Path("rt_start_" + fmt)
    dc.export(healthy, fmt, start)  # the first file set, in this format
    first = dc.load(start, format=fmt)  # ingest
    dc.export(first, fmt, "rt_again_" + fmt)  # export
    again = dc.load("rt_again_" + fmt, format=fmt)  # ingest again
    same_rows = sorted(
        a.to_row()["ann_id"] + json.dumps(a.bbox_xyxy_abs) for a in first.annotations
    ) == sorted(a.to_row()["ann_id"] + json.dumps(a.bbox_xyxy_abs) for a in again.annotations)
    trips[fmt] = (first.manifest.version, again.manifest.version, same_rows)
    print(
        f"{fmt:<13}first {first.manifest.version[:12]}...  after export and ingest {again.manifest.version[:12]}...  identical: {first.manifest.version == again.manifest.version}, boxes bit-identical: {same_rows}"
    )
record(
    "20 round trip keeps the version id",
    "ingest, export, ingest gives the same version id for all 5 formats",
    all(a == b and rows for a, b, rows in trips.values()),
    "; ".join(f"{k} {'same' if v[0] == v[1] else 'CHANGED'}" for k, v in trips.items()),
)

yolo         first d9105ffb854e...  after export and ingest d9105ffb854e...  identical: True, boxes bit-identical: True


coco         first 43144731141d...  after export and ingest 43144731141d...  identical: True, boxes bit-identical: True


voc          first 10fd63bcc3e9...  after export and ingest 10fd63bcc3e9...  identical: True, boxes bit-identical: True


cvat_xml     first 5de118782335...  after export and ingest 5de118782335...  identical: True, boxes bit-identical: True


label_studio first d48477419791...  after export and ingest d48477419791...  identical: True, boxes bit-identical: True


**What this shows.** For all five formats the version id after export and ingest is the same as before, and every box is bit-for-bit identical, so a changed id really means changed data. For YOLO this works because the reader rounds each box to exactly what the text file can hold (7 decimals of the box position, a change of at most 0.00004 pixel) when it ingests; without that rounding, boxes snapped onto the photo edge drifted a little with every pass.

## Summary: expected against actual

Every row below was filled in by the cell that ran the case: *expected* is written before the run, *actual* is generated from the real exit code and message, and the last column is an automatic check that the observed behaviour matches the
expectation (not just the exit code but also the key words of the message). A row would read `MISMATCH` if reality disagreed with the expectation.

In [46]:
def cell(text):
    return str(text).replace("|", "/").replace("\n", " ")


table = ["| Case | Expected | Actual | Check |", "|---|---|---|---|"]
table += [
    f"| {cell(case)} | {cell(expected)} | {cell(actual)} | {check} |"
    for case, expected, actual, check in ROWS
]
display(Markdown("\n".join(table)))
mismatches = [row[0] for row in ROWS if row[3] != "ok"]
print(
    f"{len(ROWS)} cases, {len(ROWS) - len(mismatches)} as expected, {len(mismatches)} mismatches",
    mismatches or "",
)

| Case | Expected | Actual | Check |
|---|---|---|---|
| 1 zero-byte image | ingest refuses (exit 1) and names the file | exit 1; error: 000000000034.jpg: unsupported or invalid image format | ok |
| 2 truncated JPEG | the plain audit warns truncated_file (a gate on warn fails, exit 1); the model audit fails closed (exit 1) | ingest exit 0; plain audit exit 0 with 1 truncated_file warning; gate on warn exit 1; model audit exit 1 | ok |
| 3 HTML saved as .jpg | ingest refuses (exit 1) and names the file | exit 1; error: 000000000034.jpg: unsupported or invalid image format | ok |
| 4 missing image file | plain ingest goes on but counts it; --strict fails (exit 1) | plain exit 0; strict exit 1: error: conversion coco -> dcp is lossy: annotations.of_missing_image=8, image... | ok |
| 5a '..' in a file name | refused (exit 1) as an unsafe path, never opened | exit 1; error: unsafe path '../secret.jpg': must be relative and stay inside the dataset | ok |
| 5b absolute path | refused (exit 1) as an unsafe path, never opened | exit 1; error: unsafe path '<work>/secret.jpg': must be relative and stay inside the dataset | ok |
| 5c symbolic link | refused (exit 1) as an unsafe path, never opened | exit 1; error: unsafe path '000000000009.jpg': resolves outside the dataset | ok |
| 5d dataset moved from its photos | audit fails closed (exit 1): photos not examined | exit 1; critical: {'external_not_checked': 1, 'images_not_hashed': 1} | ok |
| 6 box outside the image | ingest refuses (exit 1) naming the box | exit 1; error: invalid dataset: ann_id 'instances_train:1' box (10.0, 10.0, 5010.0, 60.0) outsi... | ok |
| 7 zero-area box | ingest refuses (exit 1) | exit 1; error: annotation 1: row: Value error, box must satisfy 0 <= x_min < x_max and 0 <= y_m... | ok |
| 8 NaN coordinate in COCO JSON | ingest refuses (exit 1) as invalid JSON | exit 1; error: instances_train.json is not valid JSON | ok |
| 9 duplicate box + two classes on one box | warnings duplicate_box and conflicting_class; gate on warn fails (exit 1) | exit 1; rules ['conflicting_class', 'duplicate_box'] | ok |
| 10 5000-character class name | accepted; every report cuts the name to the documented 100 or 200 characters | exit 0/0; longest name shown: 200 characters | ok |
| 11 empty dataset | ingest refuses (exit 1); the Python audit raises an error | exit 1; error: no images found in the input; API ValidationError: dataset has no images;  | ok |
| 12a same content twice in one split | ingest refuses (exit 1) naming both files | exit 1; error: images '000000000009.jpg' and 'copy_of_first.jpg' have ident... | ok |
| 12b same content across splits | ingest refuses (exit 1) naming both files | exit 1; error: images '000000000009.jpg' and 'copy_across.jpg' have identic... | ok |
| 13 file changed after ingest | CRITICAL content_changed names the file; gate fails (exit 1) | exit 1; 1 CRITICAL content_changed (000000000049.jpg) | ok |
| 14a WebP image | refused (exit 1) naming the file, never skipped; nothing written | exit 1; error: extra_photo.webp: unsupported image format (.webp); convert it to JPEG, PNG, BMP... | ok |
| 14b 100-megapixel PNG | model audit fails the gate (exit 1) with a not-embedded finding | ingest 0, plain audit 0, model audit exit 1; reason {'too_many_pixels': 1} | ok |
| 15a XML entity bomb (CVAT) | rejected quickly (exit 1, under 5 s) | exit 1 in 0.3 s; error: val.xml: DOCTYPE and entity declarations are not a... | ok |
| 15b plain DOCTYPE (CVAT) | rejected (exit 1) | exit 1; error: val.xml: DOCTYPE and entity declarations are not allowed in ... | ok |
| 16 Label Studio privacy | e-mails and signed URLs appear nowhere; the skip is counted; the null-score prediction stays out of the checks | exit 0; secrets found: 0; dropped fields named: 7 | ok |
| 17 lossy conversion (YOLO to CVAT) | --strict fails (exit 1) naming images.group_id and writes nothing | strict exit 1: error: conversion dcp -> cvat_xml is lossy: images.group_id=128; loss counts {'images.group_id': 128} | ok |
| 18 unwritable output folder | exit 1 with one clear error line naming the file | exit 1; error: Permission denied: new.dcp | ok |
| 19 model licence not allowed | exit 1 before any download; allowed licence works (exit 0) | refused exit 1/1, files downloaded: 0; with Apache-2.0 allowed exit 0 | ok |
| 20 round trip keeps the version id | ingest, export, ingest gives the same version id for all 5 formats | yolo same; coco same; voc same; cvat_xml same; label_studio same | ok |

26 cases, 26 as expected, 0 mismatches 


**What this shows.** All the defects were either refused with a clear message, counted and reported, or turned into a failing gate: none was silently accepted. The line under the table counts the cases that behaved as expected.